# Vizualisation of data INMET
source: [INMET :: BDMEP](https://bdmep.inmet.gov.br/) 
  
- *weather data*
- *whole country datasets*
  


In [51]:
import pandas as pd
from pathlib import Path 

In [52]:
current_dir = Path(Path.cwd()).parent.parent
data_dir = current_dir / "data"
source_data_dir = data_dir / "source"
print(data_dir)

c:\Users\pedro\DEV\dengue_prediction\dengue_prediction\data


In [53]:

import re
import unicodedata
 
def show_columns(df, n_range = 7):
    df = list(df.columns)
    l = []
    range = n_range
    for c in df:
        if range == n_range:
            print(l)
            l = []
            range = 0
        range += 1
        l.append(c)
    print("\n")

def normalize_column(col):
    # tira acentos
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("utf-8")
    # minúsculas
    col = col.lower()
    # troca qualquer coisa que não seja letra ou número por _
    col = re.sub(r"[^a-z0-9]+", "_", col)
    # remove _ no começo e no fim
    col = col.strip("_")
    return col

In [54]:
def prep_show(df):
    # remove coluns with more than 40% missing values
    df = df.loc[
        :, df.isnull().mean() < 0.4
    ]
    df.columns = [normalize_column(col) for col in df.columns]

    show_columns(df=df)
    print(f"porcentagem de valores ausentes:\n{(df.isnull().mean() * 100).sort_values(ascending=False)}%")
    return df

In [55]:
inmet_dir = source_data_dir / "INMET"

In [ ]:
from collections import Counter

col_counter = Counter()
file_columns = {}
file_info = []
missing_by_file = []
missing_by_file_after_window = []

for year in [2023, 2024, 2025]:
    year_dir = inmet_dir / str(year)

    for path in year_dir.iterdir():

        df = pd.read_csv(
                path,
                encoding="latin1",
                sep=";",
                skiprows=8
            )

        # Remove coluna vazia gerada por ; no final da linha, se existir
        df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

        cols = tuple(df.columns)

        file_columns[path] = cols
        col_counter.update(cols)

        # Calcula percentual de faltantes por coluna naquele arquivo
        missing_percent = df.isna().mean() * 100
        missing_by_file.append(missing_percent)

        # Guarda informações do arquivo
        file_info.append({
            "ano": year,
            "arquivo": path.name,
            "caminho": path,
            "qtd_linhas": df.shape[0],
            "qtd_colunas": df.shape[1]
        })

        # Calcula percentual de faltantes por coluna naquele arquivo depois de tratar alguns valores ausantes com janela movel
        num_cols = [
            col for col in cols
            if col not in ["Data", "Hora UTC"]
        ]
        for col in num_cols:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.replace(",", ".", regex=False)
            )

            df[col] = pd.to_numeric(df[col], errors="coerce")
        df[num_cols] = df[num_cols].fillna(
            df[num_cols].rolling(window=4, min_periods=2).mean()
        )
        missing_percent = df.isna().mean() * 100
        missing_by_file_after_window.append(missing_percent)
        
col_count_df = (
    pd.DataFrame(col_counter.items(), columns=["coluna", "qtd_arquivos"])
    .sort_values("qtd_arquivos", ascending=False)
)

file_info_df = pd.DataFrame(file_info)

print(file_info_df)
col_count_df

,coluna,qtd_arquivos
0,Data,1726
1,Hora UTC,1726
2,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",1726
3,"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORAR...",1726
4,PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),1726
5,PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),1726
6,RADIACAO GLOBAL (Kj/m²),1726
7,"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",1726
8,TEMPERATURA DO PONTO DE ORVALHO (°C),1726
9,TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),1726


In [57]:
# Porcentagem de valores faltantes
missing_df = pd.DataFrame(missing_by_file)
missing_mean_df = (
    missing_df
    .mean()
    .reset_index()
)
missing_mean_df.columns = ["coluna", "media_percentual_faltantes"]
missing_mean_df = missing_mean_df.sort_values(
    "media_percentual_faltantes",
    ascending=False
)

missing_mean_df

,coluna,media_percentual_faltantes
6,RADIACAO GLOBAL (Kj/m²),51.389976
2,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",29.368210
17,"VENTO, RAJADA MAXIMA (m/s)",27.127152
16,"VENTO, DIREÇÃO HORARIA (gr) (° (gr))",27.122544
12,TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),27.115939
18,"VENTO, VELOCIDADE HORARIA (m/s)",27.101476
11,TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),27.069205
8,TEMPERATURA DO PONTO DE ORVALHO (°C),26.896820
13,UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),26.495197
14,UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),26.469490


In [59]:
# Porcentagem de valores faltantes
missing_after_df = pd.DataFrame(missing_by_file_after_window)
missing_after_mean_df = (
    missing_after_df
    .mean()
    .reset_index()
)
missing_after_mean_df.columns = ["coluna", "media_percentual_faltantes"]
missing_after_mean_df = missing_after_mean_df.sort_values(
    "media_percentual_faltantes",
    ascending=False
)

missing_after_mean_df

,coluna,media_percentual_faltantes
6,RADIACAO GLOBAL (Kj/m²),45.530658
2,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",28.574555
17,"VENTO, RAJADA MAXIMA (m/s)",26.171827
18,"VENTO, VELOCIDADE HORARIA (m/s)",26.153241
16,"VENTO, DIREÇÃO HORARIA (gr) (° (gr))",26.145530
12,TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),25.805211
11,TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),25.755156
8,TEMPERATURA DO PONTO DE ORVALHO (°C),25.580403
13,UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),25.148203
14,UMIDADE REL. MIN. NA HORA ANT. (AUT) (%),25.136743
